In [ ]:
from steer_core.DataManager import DataManager

from steer_materials import (
    SeparatorMaterial, 
    InsulationMaterial, 
    CurrentCollectorMaterial,
    TapeMaterial, 
    AnodeMaterial,
    CathodeMaterial,
    ConductiveAdditive,
    Binder
)

# Simplified imports using __init__.py
from steer_opencell_design import (
    TablessCurrentCollector,
    AnodeFormulation,
    CathodeFormulation,
    Cathode, 
    Anode,
    Separator,
    Laminate,
    WoundJellyRoll,
    RoundMandrel,
    Tape,
    __version__
)

import pandas as pd
from datetime import datetime as dt

In [ ]:
db = DataManager()

In [ ]:
db.remove_data(table_name='cells', condition="name = 'UniGrid-NCO32140'")

In [ ]:
db.get_table_names()

In [ ]:
# Get current collector materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation_material = InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = SeparatorMaterial.from_database("Polypropylene")
tape_material = TapeMaterial.from_database("Kapton")

In [ ]:
# Create the cathode

cathode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=130,
    length=2635,
    coated_width=124,
    insulation_width=2.5,
    thickness=12
)


cathode_active_material = CathodeMaterial.from_database("NaNiMn P2-O3 Composite")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=2.64,
    mass_loading=18.18,
    insulation_material=insulation_material,
    insulation_thickness=3
)


In [ ]:
# Create the anode

anode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=134,
    length=2640,
    coated_width=128,
    insulation_width=2.5,
    thickness=12
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor B)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.05,
    mass_loading=9,
    insulation_material=insulation_material,
    insulation_thickness=3
)


In [ ]:
# make the layup

top_separator = Separator(
    material=separator_material,
    width=135,
    length=2800,
    thickness=25
)

bottom_separator = Separator(
    material=separator_material,
    width=135,
    length=2800,
    thickness=25
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_separator,
    name="UniGrid-NCO32140",
)


In [ ]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=5,
    length=500,
)

tape = Tape(
    material = tape_material,
    thickness=30
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    tape=tape,
    additional_tape_wraps=3,
    name="UniGrid-NCO32140",
)


In [ ]:
my_jellyroll.roll_properties

In [ ]:
my_jellyroll.get_spiral_plot(height=1300, width=1300).show()

In [ ]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_jellyroll.name],
    'object': [my_jellyroll.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [ ]:
db.get_data('cells')